# MoneyPrint V4 — Colab: teste real, prévia e vídeo final

Versão corrigida para **primeiro testar a voz e uma prévia de baixa resolução**, sem gastar recursos em Full HD antes de saber se o áudio funciona.

1. Execute as células **de cima para baixo** ou use **Ambiente de execução → Executar tudo**.
2. Ouça a amostra de voz antes de gerar o MP4.
3. Gere uma prévia 960×540, veja o relatório e só depois habilite `FULL_HD`.
4. Baixe o ZIP ou, se desejar, copie os arquivos para seu Google Drive.

Motor básico R$ 0 / self-hosted / open source. **Colab gratuito não garante GPU**; o teste básico com Paradee usa CPU. `qwen3_design` é um experimento GPU opcional (sem referência de voz) que pode exigir memória maior e não tem validação no Colab. Remotion não faz parte deste notebook simplificado. Não publica automaticamente nem usa APIs pagas.

## 1. Configurar ambiente e diagnosticar dependências

O notebook instala FFmpeg, fonetização PT-BR, ONNX e Pillow. Se falhar, o erro aparece nesta célula com o comando responsável. O código é obtido diretamente da branch de teste.

In [ ]:
# @title 1 — Preparar ambiente
import os, sys, json, time, shutil, subprocess
from pathlib import Path

def run_checked(args, **kwargs):
    print("EXEC:", " ".join(str(x) for x in args[:6]), flush=True)
    subprocess.run([str(x) for x in args], check=True, **kwargs)

REPO = Path("/content/moneyPrintTest")
BRANCH = "feat/moneyprint-v4-colab-lab"
print("Python:", sys.version.split()[0])
print("CUDA:", subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader") if shutil.which("nvidia-smi") else "indisponível — CPU")
run_checked(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL)
run_checked(["apt-get", "install", "-y", "-qq", "ffmpeg", "espeak-ng", "libespeak-ng1",
             "fonts-lato", "fonts-dejavu-core"], stdout=subprocess.DEVNULL)
run_checked([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary",
             "numpy>=1.26,<3", "onnxruntime>=1.19,<2", "soundfile>=0.12,<1",
             "scipy>=1.11", "misaki[pt]", "espeakng-loader", "phonemizer>=3.3,<4",
             "huggingface-hub>=0.25,<2", "pillow>=10,<12"])
if not (REPO / ".git").exists():
    run_checked(["git", "clone", "--depth", "1", "--branch", BRANCH,
                 "https://github.com/nutricionistaalmeidavh-spec/moneyPrintTest.git", str(REPO)])
else:
    run_checked(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH])
    run_checked(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from misaki.espeak import EspeakG2P
from scripts.paradee_voice import phonemes_from_g2p_result
phonemes = phonemes_from_g2p_result(EspeakG2P(language="pt-br")("Olá, vamos testar a nossa narração."))
assert phonemes
print("✅ Dependências e fonetização PT-BR verificadas.")
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Escolher roteiro, voz e formato

Para **testar primeiro**, deixe o exemplo curto. Para um vídeo completo, substitua o roteiro pelo conteúdo desejado.

- `paradee`: roda em CPU, mais leve; é o caminho já testado no Actions.
- `qwen3_design`: cria uma voz expressiva via GPU sem áudio de referência; experimental.
- `qwen3`: clona voz de referência autorizada, exigindo WAV e transcrição.

`FULL_HD` vem **desligado** inicialmente para evitar travar a sessão antes da prévia.

In [ ]:
# @title 2 — Configurar vídeo
TOPIC = "MoneyPrint — teste de produção" # @param {type:"string"}
SCRIPT = """Você sabia que é possível criar vídeos usando ferramentas de código aberto?
O MoneyPrint organiza o roteiro, produz a narração e prepara cenas para revisão."""

ASPECT = "16:9" # @param ["16:9", "9:16", "1:1"]
VOICE_ENGINE = "qwen3_design" # @param ["paradee", "qwen3_design", "qwen3"]
VOICE_SPEED = 1.06 # @param {type:"number"}
VOICE_DESCRIPTION = "Brazilian Portuguese, young adult male, warm energetic narrator, natural conversational tone, clear articulation without robotic pauses." # @param {type:"string"}
QWEN_REFERENCE_AUDIO = "" # @param {type:"string"}
QWEN_REFERENCE_TEXT = "" # @param {type:"string"}
FULL_HD = False # @param {type:"boolean"}
GENERATE_SHORTS = False # @param {type:"boolean"}
COPY_TO_DRIVE = False # @param {type:"boolean"}

WORK = Path("/content/MoneyPrintV4")
WORK.mkdir(parents=True, exist_ok=True)
print("Projeto:", TOPIC, "| voz:", VOICE_ENGINE, "| formato:", ASPECT)
if VOICE_ENGINE.startswith("qwen3"):
    print("Modo experimental GPU: possíveis limitações de VRAM; não troca automaticamente a voz.")
    run_checked([sys.executable, "-m", "pip", "install", "-q", "qwen-tts"])
if VOICE_ENGINE == "qwen3" and not (QWEN_REFERENCE_AUDIO and QWEN_REFERENCE_TEXT):
    raise ValueError("A clonagem Qwen3 exige WAV autorizado e transcrição. Use qwen3_design se não houver referência.")

## 3. Gerar e ouvir apenas a narração

Primeiro salvamos o storyboard e o WAV. Caso exista problema no modelo de voz, **nenhuma renderização de vídeo terá sido iniciada**.

In [ ]:
# @title 3 — Narração PT-BR
from scripts.colab_v4 import create_board, create_narration
from IPython.display import display, Audio
board = create_board(TOPIC, SCRIPT, aspect=ASPECT, visual_style="light_motion")
(WORK / "storyboard_inicial.json").write_text(json.dumps(board, ensure_ascii=False, indent=2), encoding="utf-8")
t0 = time.perf_counter()
board, narration, speech_report = create_narration(
    board, WORK, engine=VOICE_ENGINE, speed=VOICE_SPEED,
    reference_audio=QWEN_REFERENCE_AUDIO or None,
    reference_text=QWEN_REFERENCE_TEXT or None,
    voice_instruction=VOICE_DESCRIPTION if VOICE_ENGINE == "qwen3_design" else None)
print("Áudio pronto em", round(time.perf_counter()-t0,1), "s",
      "| duração:", round(board["total_frames"]/board["fps"], 2), "s")
print("WAV:", narration)
display(Audio(filename=str(narration)))

## 4. Gerar prévia leve e verificar MP4

A prévia usa **960×540** (ou 540×960 para Shorts), com o mesmo conteúdo e narração. Serve para detectar falhas antes da exportação final. O renderizador continua em CPU; a GPU é usada pelo TTS compatível, não automaticamente pelo Pillow/FFmpeg.

In [ ]:
# @title 4 — Prévia e QA
from scripts.colab_v4 import render_incremental, audiovisual_qa
from IPython.display import Video
PREVIEW_SIZE = {"16:9":(960,540), "9:16":(540,960), "1:1":(600,600)}[ASPECT]
t0 = time.perf_counter()
preview_result = render_incremental(
    board, narration, REPO, WORK / "preview",
    cache_dir=WORK / "cache_preview",
    width=PREVIEW_SIZE[0], height=PREVIEW_SIZE[1])
preview_path = Path(preview_result["output"])
preview_qa = audiovisual_qa(preview_path, board, WORK / "preview",
                            expected_size=PREVIEW_SIZE)
print("Tempo:", round(time.perf_counter()-t0,1), "s")
print(json.dumps(preview_qa, ensure_ascii=False, indent=2))
if not preview_qa["pass"]:
    raise RuntimeError("QA da prévia reprovado. Verifique as pausas e a mídia antes do Full HD.")
display(Video(filename=str(preview_path), embed=True, width=640))
print("Prévia disponível:", preview_path)

## 5. Full HD somente depois da prévia

Se quiser a versão final, marque `FULL_HD=True` na célula de configuração e execute esta célula. Caso contrário, o pacote final inclui a prévia validada.

**Aviso:** o `light_motion` usa composição Pillow/FFmpeg em CPU. Full HD pode ser demorado, mesmo com GPU T4.

In [ ]:
# @title 5 — Versão final (opcional)
from scripts.colab_v4 import ALLOWED_ASPECTS
if FULL_HD:
    t0 = time.perf_counter()
    final_result = render_incremental(
        board, narration, REPO, WORK / "final",
        cache_dir=WORK / "cache_fullhd")
    movie = Path(final_result["output"])
    final_qa = audiovisual_qa(movie, board, WORK / "final")
    print("Full HD levou", round(time.perf_counter()-t0,1), "s")
    print(json.dumps(final_qa, ensure_ascii=False, indent=2))
    if not final_qa["pass"]:
        raise RuntimeError("QA Full HD reprovado. Não usar como vídeo final.")
else:
    movie = preview_path
    print("Prévia aprovada selecionada. Para Full HD, habilite FULL_HD na configuração e execute esta célula.")

## 6. Shorts verticais e legenda (opcionais)

Só serão gerados se você solicitar explicitamente. Faster-whisper roda localmente e pode baixar modelo adicional, sem API paga.

In [ ]:
# @title 6 — Shorts (opcional)
shorts = []
if GENERATE_SHORTS:
    run_checked([sys.executable, "-m", "pip", "install", "-q", "faster-whisper>=1.1,<2"])
    from scripts.colab_v4 import transcribe_words, export_generic_shorts
    transcript = transcribe_words(movie, WORK / "qa", model_size="tiny")
    shorts = export_generic_shorts(movie, WORK / "shorts", transcript, count=3)
    print("Shorts produzidos:", len(shorts))
else:
    print("Shorts desabilitados: nenhum processamento adicional.")

## 7. Baixar vídeo, WAV e relatórios

O ZIP contém a prévia ou o Full HD escolhido, o áudio e os metadados. O Google Drive **só é acessado se você habilitar a opção**, com autorização na sua própria sessão. Se o navegador do iPhone não iniciar o download, os arquivos permanecem em `/content/MoneyPrintV4` até a sessão terminar.

In [ ]:
# @title 7 — Exportar pacote
import zipfile
bundle = WORK / "MoneyPrint_V4_Resultado.zip"
include = [movie, narration, WORK / "storyboard_inicial.json",
           WORK / "storyboard_sincronizado.json",
           WORK / "voz_diagnostico.json",
           (WORK / ("final" if FULL_HD else "preview") / "qa_audio_video.json")]
include.extend(Path(item["path"]) for item in shorts)
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for p in include:
        if p.is_file():
            archive.write(p, arcname=p.relative_to(WORK))
print("Pacote criado:", bundle, "| bytes:", bundle.stat().st_size)
if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/MoneyPrintV4")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy2(bundle, dest / bundle.name)
    print("Cópia no Drive:", dest / bundle.name)
from google.colab import files
files.download(str(bundle))